In [1]:
import netCDF4 as nc
import numpy as np
import torch

def sww_to_depth_tensor_gpu(sww_file, nx=150, ny=150, min_depth_threshold=0.01, device='cuda'):
    """
    Interpolates SWW data to a uniform grid of size (nx, ny) and extracts elevation.
    Target resolution is inferred from the bounding box and the specified nx/ny.
    """

    # 1. Load data
    ds = nc.Dataset(sww_file, 'r')
    x_raw = ds.variables['x'][:]
    y_raw = ds.variables['y'][:]
    elevation_raw = ds.variables['elevation'][:]  
    stage_all_raw = ds.variables['stage'][:]
    ds.close()

    # 2. Calculate Depth (NumPy)
    depth_all_raw = stage_all_raw - elevation_raw
    depth_all_raw = np.nan_to_num(depth_all_raw, nan=0.0)
    depth_all_raw[depth_all_raw < min_depth_threshold] = 0.0
    nt, num_nodes = depth_all_raw.shape

    # 3. Transfer to GPU
    points_gpu = torch.stack([
        torch.from_numpy(x_raw).to(device).float(),
        torch.from_numpy(y_raw).to(device).float()
    ], dim=1)
    
    depth_gpu = torch.from_numpy(depth_all_raw).to(device).float()
    elev_gpu = torch.from_numpy(elevation_raw).to(device).float()

    # 4. Create Uniform Target Grid based on nx and ny
    x_min, x_max = x_raw.min(), x_raw.max()
    y_min, y_max = y_raw.min(), y_raw.max()

    # Using steps=nx and steps=ny to create the linspace directly
    lin_x = torch.linspace(x_min, x_max, steps=nx, device=device)
    lin_y = torch.linspace(y_min, y_max, steps=ny, device=device)
    
    # Generate grid coordinates
    mesh_y, mesh_x = torch.meshgrid(lin_y, lin_x, indexing='ij')
    grid_coords = torch.stack([mesh_x.flatten(), mesh_y.flatten()], dim=1)

    # 5. Find Nearest Neighbors (Chunked)
    num_grid_points = nx * ny
    nearest_indices = torch.zeros(num_grid_points, dtype=torch.long, device=device)
    
    chunk_size = 5000 
    for i in range(0, num_grid_points, chunk_size):
        end_i = min(i + chunk_size, num_grid_points)
        dists = torch.cdist(grid_coords[i:end_i], points_gpu) 
        nearest_indices[i:end_i] = torch.argmin(dists, dim=1)
        
    # 6. Map Depth (all timesteps) and Elevation (static)
    depth_interp_flat = depth_gpu[:, nearest_indices]
    elev_interp_flat = elev_gpu[nearest_indices]
    
    # 7. Reshape and Permute to maintain original [nx, ny, nt] and [nx, ny] output
    tensor_depth = depth_interp_flat.view(nt, ny, nx).permute(2, 1, 0)
    tensor_elev = elev_interp_flat.view(ny, nx).T
    
    grid_x_2d = mesh_x.T # [nx, ny]
    grid_y_2d = mesh_y.T # [nx, ny]
    
    return tensor_depth, tensor_elev, grid_x_2d, grid_y_2d

In [2]:
import torch
import netCDF4 as nc
import numpy as np
import os

def generate_gaussian_discharge_tensor(tms_dir, grid_x_2d, grid_y_2d, radius=500.0, time_factor=6, device='cuda'):
    """
    Creates a unified spatio-temporal tensor for discharge (Q) using 2D Gaussian kernels
    and Temporal Block Summation with padding to ensure specific output dimensions.
    
    Inputs:
    -------
    - tms_dir (str): Directory containing gauge .tms files.
    - grid_x_2d/grid_y_2d (torch.Tensor): 2D coordinate matrices [nx, ny].
    - gauges (list): Metadata for gauge locations and IDs.
    - radius (float): Spatial influence cutoff.
    - time_factor (int): Factor to reduce the time dimension (e.g., 6).
    - device (str): Target device ('cuda' or 'cpu').

    Outputs:
    --------
    - q_tensor (torch.Tensor): Shape [nx, ny, ceil(nt / time_factor)].
    - aggregated_time (np.array): Reduced timeline.
    """
    CENTER_EASTING  = 3100.0
    CENTER_NORTHING = 17600.0

    gauges = [
        {"id": "Bc1_1",  "x": CENTER_EASTING, "y": CENTER_NORTHING},
    ]


    nx, ny = grid_x_2d.shape
    x_offset = 0
    y_offset = 0
    sigma = radius / 2.0 

    # 1. TEMPORAL SYNCHRONIZATION
    master_time = None
    for gauge in gauges:
        usgs_id = gauge['id'].split('_')[-1]
        tms_path = os.path.join(tms_dir, f"{usgs_id}.tms")
        if os.path.exists(tms_path):
            with nc.Dataset(tms_path, 'r') as ds:
                master_time = ds.variables['time'][:].copy()
            break 
            
    if master_time is None:
        raise FileNotFoundError("Could not find any .tms files.")
    
    nt_orig = len(master_time)
    # 2. PADDING LOGIC (To reach 45 steps from 265)
    # Calculate how many extra steps are needed to be divisible by time_factor
    remainder = nt_orig % time_factor
    padding_size = (time_factor - remainder) if remainder > 0 else 0
    nt_padded = nt_orig + padding_size
    nt_new = nt_padded // time_factor # This will be 45 for nt_orig=265, factor=6

    # 3. TENSOR ALLOCATION (Allocate with padded size)
    q_tensor = torch.zeros((nx, ny, nt_padded), device=device)

    # 4. KERNEL GENERATION AND DATA INJECTION
    for gauge in gauges:
        usgs_id = gauge['id'].split('_')[-1]
        tms_path = os.path.join(tms_dir, f"{usgs_id}.tms")
        
        if not os.path.exists(tms_path):
            continue

        with nc.Dataset(tms_path, 'r') as ds:
            q_values = ds.variables['discharge'][:]
        
        q_gpu = torch.from_numpy(q_values).to(device).float()
        local_gx = gauge['x'] - x_offset
        local_gy = gauge['y'] - y_offset

        # 5. SPATIAL WEIGHTING
        dist_sq = (grid_x_2d - local_gx)**2 + (grid_y_2d - local_gy)**2
        weights = torch.exp(-dist_sq / (2 * sigma**2))
        weights[dist_sq > radius**2] = 0 
        
        if weights.sum() > 0:
            weights = weights / weights.sum()

        # 6. INJECTION (Inject original values into the start of the padded tensor)
        # The remaining steps (266-270) remain zero-filled
        q_tensor[..., :nt_orig] += weights.unsqueeze(-1) * q_gpu.unsqueeze(0).unsqueeze(0)

    # 7. TEMPORAL BLOCK SUMMATION
    # Reshape and sum along the block dimension
    q_tensor = q_tensor.view(nx, ny, nt_new, time_factor).sum(dim=-1)

    # 8. TIME AGGREGATION
    # Pad master_time with the last timestamp to match length before reshaping
    if padding_size > 0:
        last_time = master_time[-1]
        time_padding = np.full(padding_size, last_time)
        master_time_padded = np.concatenate([master_time, time_padding])
    else:
        master_time_padded = master_time

    aggregated_time = master_time_padded.reshape(nt_new, time_factor).mean(axis=-1)

    return q_tensor, aggregated_time

In [3]:
import os
import torch
import re
import numpy as np
from tqdm import tqdm

def process_root_to_depth_tensors(root_dir, nx=328, ny=164, threshold=0.025, q_radius=500.0, time_factor=6, if_UQ=False):
    """
    Processes scenarios by pre-allocating CPU memory to prevent 'cat' memory spikes.
    Static data (grid, elevation, time) is stored only once.
    """
    # 1. First Pass: Find all valid scenarios to determine 'nbatch'
    sww_pattern = re.compile(r"Dam_Break_group_.*_scenario_.*_merged\.sww$")
    scenario_list = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if sww_pattern.match(file):
                tms_dir = os.path.join(root, 'tms_files')
                if os.path.exists(tms_dir):
                    scenario_list.append((os.path.join(root, file), tms_dir))
    
    nbatch = len(scenario_list)
    print(f'nbatch: {nbatch}')
    if nbatch == 0:
        return None

    # 2. Get dimensions from the first scenario to pre-allocate
    first_sww, first_tms = scenario_list[0]
    d_sample, e_sample, gx_sample, gy_sample = sww_to_depth_tensor_gpu(first_sww, nx=nx, ny=ny, min_depth_threshold=threshold)
    q_sample, t_sample = generate_gaussian_discharge_tensor(first_tms, gx_sample, gy_sample, radius=q_radius, time_factor=time_factor)
    
    nx, ny, nt = d_sample.shape
    
    # 3. Pre-allocate dynamic tensors on CPU
    # This allocates the memory ONCE. No more growing lists or 'cat' operations.
    batch_depth = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    batch_elev = torch.zeros((nbatch, nx, ny), dtype=torch.float32)
    # batch_q = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    
    # Store static data only once
    # final_elevation = e_sample.cpu()
    final_gx = gx_sample.cpu()
    final_gy = gy_sample.cpu()
    final_time = torch.from_numpy(t_sample).float()

    print(f"Pre-allocated CPU memory for {nbatch} scenarios ({nx}x{ny}x{nt})")

    # 4. Second Pass: Fill the pre-allocated tensors

    for i, (sww_path, tms_dir) in enumerate(tqdm(scenario_list, desc="Processing Scenarios")):
        try:
            # Process on GPU for speed
            depth, elve, gx, gy = sww_to_depth_tensor_gpu(sww_path, nx=nx, ny=ny, min_depth_threshold=threshold)
            # q_tensor, _ = generate_gaussian_discharge_tensor(tms_dir, gx, gy, radius=q_radius, time_factor=time_factor)
            
            # Direct assignment to the CPU tensor
            batch_depth[i] = depth.cpu()
            if if_UQ:
                batch_elev[i] = elve.cpu()
            # batch_q[i] = q_tensor.cpu()
            
            # Explicitly clean up GPU memory after each scenario
            del (depth, 
                # q_tensor, 
                gx, 
                gy)
            
            if if_UQ:
                del elve

            torch.cuda.empty_cache()
            
            # Note: You can remove the old "if (i+1) % 5" print statements 
            # as tqdm provides a much cleaner live update.
                
        except Exception as e:
            # Using tqdm.write prevents the error message from breaking the progress bar
            tqdm.write(f"Error processing scenario {i}: {e}")

    if if_UQ:
        final_elevation = batch_elev
    else:
        final_elevation = e_sample.cpu()

    return {
        "depth": batch_depth,      # [nbatch, nx, ny, nt]
        # "discharge": batch_q,      # [nbatch, nx, ny, nt]
        "elevation": final_elevation, 
        # "grid_x": final_gx, 
        # "grid_y": final_gy,
        # "time": final_time
    }

In [4]:
import os
import torch
import re
import numpy as np
from tqdm import tqdm

def process_root_to_q_tensors(root_dir, nx=328, ny=164, q_radius=500.0, time_factor=6):
    """
    Processes scenarios by pre-allocating CPU memory for discharge (Q) only.
    Returns the batched Q tensor and the synchronized time array.
    """
    # 1. First Pass: Find all valid scenarios that have a 'tms_files' directory
    sww_pattern = re.compile(r"Dam_Break_group_.*_scenario_.*_merged\.sww$")
    scenario_list = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if sww_pattern.match(file):
                tms_dir = os.path.join(root, 'tms_files')
                if os.path.exists(tms_dir):
                    scenario_list.append((os.path.join(root, file), tms_dir))
    
    nbatch = len(scenario_list)
    if nbatch == 0:
        print("No scenarios with 'tms_files' found.")
        return None

    # 2. Get spatial and temporal dimensions from the first scenario
    # We still need one call to sww_to_depth_tensor to get the grid (gx, gy)
    first_sww, first_tms = scenario_list[0]
    _, _, gx_sample, gy_sample = sww_to_depth_tensor_gpu(first_sww, nx=nx, ny=ny)
    
    # Generate a sample Q to determine the final 'nt' after time_factor aggregation
    q_sample, t_sample = generate_gaussian_discharge_tensor(
        first_tms, gx_sample, gy_sample, radius=q_radius, time_factor=time_factor
    )
    
    nx, ny, nt = q_sample.shape
    print(f'nx, ny, nt: {nx}, {ny}, {nt}')
    # 3. Pre-allocate dynamic Discharge tensor on CPU RAM
    batch_q = torch.zeros((nbatch, nx, ny, nt), dtype=torch.float32)
    
    # Store the synchronized time only once
    final_time = torch.from_numpy(t_sample).float()

    print(f"Pre-allocated CPU memory for {nbatch} Discharge scenarios ({nx}x{ny}x{nt})")

    # 4. Second Pass: Fill the pre-allocated Discharge tensor
    for i, (sww_path, tms_dir) in enumerate(tqdm(scenario_list, desc="Generating Q Tensors")):
        try:
            # We need the grid for each scenario to ensure spatial alignment
            _, _, gx, gy = sww_to_depth_tensor_gpu(sww_path, nx=nx, ny=ny)
            
            # Generate Gaussian Q
            q_tensor, _ = generate_gaussian_discharge_tensor(
                tms_dir, gx, gy, radius=q_radius, time_factor=time_factor
            )
            
            # Direct assignment to CPU memory
            batch_q[i] = q_tensor.cpu()
            
            # Explicitly clean up GPU memory
            del q_tensor, gx, gy
            torch.cuda.empty_cache()
                
        except Exception as e:
            tqdm.write(f"Error processing scenario {i}: {e}")

    return {
        "discharge": batch_q, # [nbatch, nx, ny, nt]
        "time": final_time     # [nt]
    }

In [5]:
# from pathlib import Path

# MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_UQ/'

# # Create the directory
# # parents=True: creates missing parent folders
# # exist_ok=True: doesn't throw an error if the folder already exists
# Path(MAIN_PATH).mkdir(parents=True, exist_ok=True)

# print(f"Path verified: {MAIN_PATH}")

In [6]:
import numpy as np
import torch
from scipy.interpolate import griddata

def fill_edge_nans(bed_tensor):
    # 1. Convert to numpy for Scipy processing
    bed_np = bed_tensor.cpu().numpy()
    
    # 2. Identify indices of valid data and NaN data
    # 'y' and 'x' coordinates of all pixels
    y, x = np.indices(bed_np.shape)
    
    # Mask of valid (not NaN) pixels
    mask_valid = ~np.isnan(bed_np)
    
    # Points where we have data (input for interpolation)
    points_valid = np.stack((y[mask_valid], x[mask_valid]), axis=1)
    values_valid = bed_np[mask_valid]
    
    # Points where we need to fill data
    points_nan = np.stack((y[~mask_valid], x[~mask_valid]), axis=1)
    
    # 3. Use 'nearest' interpolation to fill the edges
    # 'nearest' is better than 'linear' here because it avoids 
    # creating weird slopes at the very edge of your domain.
    filled_values = griddata(points_valid, values_valid, points_nan, method='nearest')
    
    # 4. Place the filled values back into the array
    bed_np[~mask_valid] = filled_values
    
    return torch.from_numpy(bed_np).to(bed_tensor.device)



In [7]:
import numpy as np
import torch
from scipy.interpolate import griddata

def fill_edge_nans(bed_tensor: torch.Tensor) -> torch.Tensor:
    """
    Fill NaNs using nearest-neighbor interpolation from existing (non-NaN) pixels.
    Supports:
      - 2D: [nx, ny]
      - 3D: [nbatch, nx, ny]  (process each batch independently)
    """
    if bed_tensor.dim() not in (2, 3):
        raise ValueError(f"Expected 2D or 3D tensor, got shape {tuple(bed_tensor.shape)}")

    device = bed_tensor.device
    out_dtype = bed_tensor.dtype

    # SciPy wants NumPy on CPU; also use float to support NaNs reliably
    bed_np = bed_tensor.detach().to("cpu").numpy()

    def _fill_2d(arr2d: np.ndarray) -> np.ndarray:
        # If there are no NaNs, return fast
        if not np.isnan(arr2d).any():
            return arr2d

        # If everything is NaN, we cannot infer anything
        if np.isnan(arr2d).all():
            return arr2d  # or raise, depending on your preference

        y, x = np.indices(arr2d.shape)
        mask_valid = ~np.isnan(arr2d)

        points_valid = np.stack((y[mask_valid], x[mask_valid]), axis=1)
        values_valid = arr2d[mask_valid]

        points_nan = np.stack((y[~mask_valid], x[~mask_valid]), axis=1)

        filled_values = griddata(
            points_valid,
            values_valid,
            points_nan,
            method="nearest",
        )

        arr2d = arr2d.copy()
        arr2d[~mask_valid] = filled_values
        return arr2d

    if bed_np.ndim == 2:
        filled_np = _fill_2d(bed_np)
        return torch.from_numpy(filled_np).to(device=device, dtype=out_dtype)

    # 3D: process per batch
    filled_np = np.empty_like(bed_np)
    for b in range(bed_np.shape[0]):
        filled_np[b] = _fill_2d(bed_np[b])

    return torch.from_numpy(filled_np).to(device=device, dtype=out_dtype)


In [8]:
# import os
# import torch

# # ==========================================
# # 1. Process and Save Target Tensors (Depth & Elevation)
# # ==========================================
# # Generate depth and static terrain data

# MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_1'

# nx=328
# ny=164

# DATA_depth = process_root_to_depth_tensors(MAIN_PATH, nx=nx, ny=ny, q_radius=1000, time_factor=1)

# # Define file paths for the solution (Target) and bed (Static) tensors
# solution_file = os.path.join(MAIN_PATH, "dam_break_processed_data_solution_fine_res.pt")
# bed_file      = os.path.join(MAIN_PATH, "dam_break_processed_data_bed_fine_res.pt")

# bed = fill_edge_nans(DATA_depth['elevation'])

# print(f"Saving Target tensors to: {MAIN_PATH}")
# torch.save(DATA_depth['depth'], solution_file)
# torch.save(bed, bed_file)

# # Explicitly clear memory before starting the next heavy process
# del DATA_depth, bed
# torch.cuda.empty_cache()


In [9]:
import os
import torch

# ==========================================
# 1. Process and Save Target Tensors (Depth & Elevation)
# ==========================================
# Generate depth and static terrain data

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_UQ2'
if_UQ = True
DATA_depth = process_root_to_depth_tensors(MAIN_PATH, q_radius=2000, time_factor=1, if_UQ=if_UQ)

# Define file paths for the solution (Target) and bed (Static) tensors
solution_file = os.path.join(MAIN_PATH, "dam_break_processed_data_solution_UQ2.pt")
bed_file      = os.path.join(MAIN_PATH, "dam_break_processed_data_bed_UQ2.pt")

bed = fill_edge_nans(DATA_depth['elevation'])

print(f"Saving Target tensors to: {MAIN_PATH}")
torch.save(DATA_depth['depth'], solution_file)
torch.save(bed, bed_file)

# Explicitly clear memory before starting the next heavy process
del DATA_depth, bed
torch.cuda.empty_cache()

# ==========================================
# 2. Process and Save Input Tensors (Discharge)
# ==========================================
# Generate the spatio-temporal discharge input

DATA_q = process_root_to_q_tensors(MAIN_PATH, q_radius=10000, time_factor=900)

# Define file path for the input tensor
input_file = os.path.join(MAIN_PATH, "dam_break_processed_data_input_UQ2.pt")

print(f"Saving Input tensors to: {MAIN_PATH}")
torch.save(DATA_q['discharge'], input_file)

# Final cleanup
del DATA_q
torch.cuda.empty_cache()

print("✅ Data processing and storage complete.")

nbatch: 500
Pre-allocated CPU memory for 500 scenarios (328x164x97)


Processing Scenarios: 100%|██████████| 500/500 [02:19<00:00,  3.57it/s]


Saving Target tensors to: /storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_UQ2
nx, ny, nt: 328, 164, 97
Pre-allocated CPU memory for 500 Discharge scenarios (328x164x97)


Generating Q Tensors: 100%|██████████| 500/500 [09:54<00:00,  1.19s/it]


Saving Input tensors to: /storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_UQ2
✅ Data processing and storage complete.


In [ ]:
import torch
import json
import os
import gc

# Define your paths as provided
MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_1/'
a_path = MAIN_PATH + "dam_break_matthew_processed_data_input.pt"
u_path = MAIN_PATH + "dam_break_processed_data_solution.pt"
bed_file = MAIN_PATH + 'dam_break_processed_data_bed.pt'

def scale_and_save_dataset_inplace(inp1_path, inp2_path, out_path, save_dir):
    """
    Loads tensors one-by-one and scales them in-place to minimize RAM usage.
    """
    states = {}
    
    # --- 1. PROCESS INPUT 1 (Q/Input) ---
    print("Processing Input 1 (Q)...")
    # Load into memory
    data = torch.load(inp1_path)
    
    # Log-Standardization: log1p_ is in-place
    data.log1p_() 
    mu1, std1 = data.mean().item(), data.std().item()
    # sub_ and div_ are in-place
    data.sub_(mu1).div_(std1 + 1e-8)
    
    # Save with required naming convention
    save_path1 = inp1_path.replace(".pt", "_scaled.pt")
    torch.save(data, save_path1)
    states['Q'] = {'method': 'log_zscore', 'mean': mu1, 'std': std1}
    
    # Clear memory explicitly
    del data
    gc.collect()

    # --- 2. PROCESS INPUT 2 (Bed) ---
    print("Processing Input 2 (Bed)...")
    data = torch.load(inp2_path)
    
    # Z-score Standardization
    mu2, std2 = data.mean().item(), data.std().item()
    data.sub_(mu2).div_(std2 + 1e-8)
    
    save_path2 = inp2_path.replace(".pt", "_scaled.pt")
    torch.save(data, save_path2)
    states['Bed'] = {'method': 'zscore', 'mean': mu2, 'std': std2}
    
    del data
    gc.collect()

    # --- 3. PROCESS OUTPUT (Solution) ---
    print("Processing Output (Solution)...")
    data = torch.load(out_path)
    
    # Z-score Standardization
    mu_out, std_out = data.mean().item(), data.std().item()
    data.sub_(mu_out).div_(std_out + 1e-8)
    
    save_path3 = out_path.replace(".pt", "_scaled.pt")
    torch.save(data, save_path3)
    states['output'] = {'method': 'zscore', 'mean': mu_out, 'std': std_out}
    
    del data
    gc.collect()

    # --- 4. SAVE SCALING STATES ---
    state_file = os.path.join(save_dir, "dam_break_scaling_states.json")
    with open(state_file, "w") as f:
        json.dump(states, f, indent=4)

    print(f"\nSuccess! All files saved in: {save_dir}")
    print(f"Scaling parameters saved to: {state_file}")
    return states

# Run the function
scaling_metadata = scale_and_save_dataset_inplace(a_path, bed_file, u_path, MAIN_PATH)

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.utils.data.distributed import DistributedSampler

import sys
sys.path.append('../..')
from lib.helper import LargeHydrologyDataset

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_UQ/'
# This creates a 'pointer' to the data on disk without filling up your RAM
a_path = MAIN_PATH + "dam_break_matthew_processed_data_input.pt"
u_path = MAIN_PATH + "dam_break_processed_data_solution_UQ.pt"
bed_file = MAIN_PATH + 'dam_break_processed_data_bed.pt'

In [ ]:
u = torch.load(u_path, mmap=True, map_location='cuda')
u.shape

In [ ]:
depth = u[u>=0.025]

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# your tensor (example - replace with your actual variable)
# tensor = torch.randn(1000, 328, 164, 97)

# Option A: Convert to numpy once (usually fastest)
values = depth[..., ::2].cpu().numpy().ravel()           # flatten to 1D vector

# Option B: if you want to avoid full copy in memory (very large tensor)
# values = tensor.cpu().flatten().numpy()

plt.figure(figsize=(10, 6))
plt.hist(values, bins=150, density=False, alpha=0.7, color='cornflowerblue', edgecolor='navy')

plt.title('Histogram of all values in tensor (shape 1000×328×164×97)')
plt.xlabel('Value')
plt.ylabel('Count')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
import torch

def print_tensor_stats(name, tensor):
    """
    Prints key statistical properties of a tensor to help debug training issues.
    """
    if not isinstance(tensor, torch.Tensor):
        print(f"{name}: Not a tensor (Type: {type(tensor)})")
        return

    # Basic stats
    mean = tensor.mean().item()
    std = tensor.std().item()
    t_min = tensor.min().item()
    t_max = tensor.max().item()
    

    print(f"--- Stats for: {name} ---")
    print(f"  Shape:  {list(tensor.shape)}")
    print(f"  Range:  [{t_min:.4f}, {t_max:.4f}]")
    print(f"  Mean:   {mean:.4f} | Std: {std:.4f}")
    print("-" * 30)

In [ ]:
u = torch.load(u_path, mmap=True, map_location='cuda')[:500]
u.shape

In [ ]:
import torch
import torch.nn.functional as F

def scale_spatial_resolution_chunked(u, N, chunk_size=50):
    """
    Interpolates [nb, nx, ny, nt] in chunks to avoid CUDA OutOfMemory.
    """
    nb, nx, ny, nt = u.shape
    new_nx, new_ny = int(nx * N), int(ny * N)
    
    # Pre-allocate output tensor on CPU or GPU 
    # (If the final result is too big for GPU, keep it on CPU for now)
    # Using the same dtype as input
    u_finer = torch.empty((nb, new_nx, new_ny, nt), dtype=u.dtype, device=u.device)

    for i in range(0, nb, chunk_size):
        # 1. Grab a chunk and move to GPU
        chunk = u[i : i + chunk_size].cuda()
        
        # 2. Permute: [batch, time, nx, ny]
        chunk = chunk.permute(0, 3, 1, 2)
        
        # 3. Interpolate
        with torch.no_grad(): # Use no_grad to save memory if not training
            interpolated_chunk = F.interpolate(
                chunk, 
                size=(new_nx, new_ny), 
                mode='area'
            )
        
        # 4. Permute back: [batch, nx_new, ny_new, nt]
        interpolated_chunk = interpolated_chunk.permute(0, 2, 3, 1)
        
        # 5. Store back into pre-allocated tensor
        u_finer[i : i + chunk_size] = interpolated_chunk.cpu()
        
        # Explicitly clear the chunk from GPU memory
        del chunk, interpolated_chunk
        torch.cuda.empty_cache()
        

    return u_finer

# Usage:


In [ ]:
u_finer.shape

In [ ]:
print_tensor_stats('output', u)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(u[15, ..., 2].cpu().numpy())


In [ ]:
plt.imshow(bed.cpu().numpy())

In [ ]:
a = torch.load(a_path, mmap=True, map_location='cuda')
a.isnan().any()

In [ ]:
print_tensor_stats('Input1', a)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(a[15, ..., 30].cpu().numpy())


In [ ]:
torch.mean(a[15]-a[165])

In [ ]:


bed = torch.load(bed_file, mmap=True, map_location='cuda')
bed.isnan().any()

In [ ]:
print_tensor_stats('Input2', bed)

In [ ]:
bed.min(), bed.max()

In [ ]:
import torch
import json
import os
import gc

def scale_and_save_dataset_memory_efficient(inp1_path, inp2_path, out_path, MAIN_PATH, base_name="processed"):
    """
    Processes large tensors one-by-one to avoid Out-Of-Memory (OOM) errors.
    """
    states = {}
    os.makedirs(MAIN_PATH, exist_ok=True)

    # --- PROCESS INPUT 1 (Q) ---
    # Load -> Log -> Mean/Std -> Scale -> Save -> Delete
    data = torch.load(inp1_path)
    data.log1p_()  # In-place log(1+x) to save memory
    mu, std = data.mean().item(), data.std().item()
    data.sub_(mu).div_(std + 1e-8)  # In-place standardization
    
    torch.save(data, os.path.join(MAIN_PATH, f"{base_name}_input_scaled.pt"))
    states['Q'] = {'method': 'log_zscore', 'mean': mu, 'std': std}
    
    del data # Explicitly remove from RAM
    gc.collect() # Force garbage collection

    # --- PROCESS INPUT 2 (Bed) ---
    data = torch.load(inp2_path)
    mu, std = data.mean().item(), data.std().item()
    data.sub_(mu).div_(std + 1e-8)
    
    torch.save(data, os.path.join(MAIN_PATH, f"{base_name}_bed_scaled.pt"))
    states['Bed'] = {'method': 'zscore', 'mean': mu, 'std': std}
    
    del data
    gc.collect()

    # --- PROCESS OUTPUT (Solution) ---
    data = torch.load(out_path)
    mu, std = data.mean().item(), data.std().item()
    data.sub_(mu).div_(std + 1e-8)
    
    torch.save(data, os.path.join(MAIN_PATH, f"{base_name}_solution_scaled.pt"))
    states['output'] = {'method': 'zscore', 'mean': mu, 'std': std}
    
    del data
    gc.collect()

    # --- SAVE STATES ---
    state_path = os.path.join(MAIN_PATH, f"{base_name}_scaling_states.json")
    with open(state_path, "w") as f:
        json.dump(states, f, indent=4)

    print(f"Success! Memory-efficient scaling complete. States saved to {state_path}")
    return states

In [ ]:
import torch
import numpy as np
import pandas as pd
from matplotlib.path import Path

def mask_tensor_by_raw_csv(tensor, csv_path, Lx, Ly):
    """
    Masks a tensor [nb, nx, ny, nt] by zeroing regions inside points from a headerless CSV.
    csv_path: path to CSV with rows like: 1250.5, 4500.2
    """
    # 1. Load coordinates from CSV (no headers)
    # Using index 0 for X and index 1 for Y
    df = pd.read_csv(csv_path, header=None)
    points = df[[0, 1]].values 
    
    nb, nx, ny, nt = tensor.shape
    device = tensor.device
    
    # 2. Create physical coordinate grid
    # Mapping [0...nx] to [0...Lx]
    x_coords = np.linspace(0, Lx, nx)
    y_coords = np.linspace(0, Ly, ny)
    xv, yv = np.meshgrid(x_coords, y_coords, indexing='ij')
    
    # Flatten grid for the point-in-polygon test
    grid_points = np.vstack((xv.flatten(), yv.flatten())).T
    
    # 3. Generate the mask using Matplotlib's Path
    # This creates a closed polygon from your boundary points
    poly_path = Path(points)
    mask_flat = poly_path.contains_points(grid_points)
    
    # 4. Convert to PyTorch and reshape
    mask_2d = torch.from_numpy(mask_flat.reshape(nx, ny)).to(device)
    
    # Invert: Points INSIDE the polygon become 0 (masked), OUTSIDE stay 1
    # .logical_not() is used for booleans, then cast to float for multiplication
    final_mask = torch.logical_not(mask_2d).float()
    
    # 5. Reshape for broadcasting [1, nx, ny, 1]
    final_mask = final_mask.view(1, nx, ny, 1)
    
    # 6. Apply to the original tensor
    return tensor * final_mask

# --- Example Usage ---
# tensor = torch.ones(1, 128, 128, 100)
# (26460.0, 20940.0)

u = mask_tensor_by_raw_csv(u, 'DEM_data/dam_up.csv', Lx=26460.0, Ly=20940.0)

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.utils.data.distributed import DistributedSampler

import sys
sys.path.append('../..')
from lib.helper import LargeHydrologyDataset

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_1/'
# This creates a 'pointer' to the data on disk without filling up your RAM
a_path = MAIN_PATH + "dam_break_matthew_processed_data_input.pt"
u_path = MAIN_PATH + "dam_break_processed_data_solution.pt"


# 1. Initialize the full dataset
full_dataset = LargeHydrologyDataset(a_path, u_path, mask=True, csv_path='DEM_data/dam_up.csv',  Lx=26460.0, Ly=20940.0)
# 2. Split with a fixed generator for reproducibility
train_size = int(0.8 * len(full_dataset))
eval_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)

train_dataset, eval_dataset = random_split(
    full_dataset, [train_size, eval_size], generator=generator
)
world_size = 5
rank = 2
batch_size = 32
# 3. Define Distributed Samplers
# rank and world_size are provided by your DDP setup (e.g., dist.get_rank())
train_sampler = DistributedSampler(
    train_dataset, 
    num_replicas=world_size, 
    rank=rank, 
    shuffle=True  # Shuffle training data across GPUs
)

eval_sampler = DistributedSampler(
    eval_dataset, 
    num_replicas=world_size, 
    rank=rank, 
    shuffle=False # Keep evaluation order consistent
)

# 4. Create DataLoaders
# CRITICAL: shuffle must be False here because the sampler is used
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=train_sampler,
    num_workers=4,
    pin_memory=True
)

eval_loader = DataLoader(
    eval_dataset, 
    batch_size=batch_size, 
    sampler=eval_sampler,
    num_workers=4,
    pin_memory=True
)

In [ ]:
# Get one batch
a_batch, u_batch = next(iter(eval_loader))   # [B, ...]

# Randomly pick 4 samples
idx = torch.randperm(a_batch.size(0))[:4]

# Unsqueeze dim=0 and collect
a_list = [a_batch[i].unsqueeze(0) for i in idx]
u_list = [u_batch[i].unsqueeze(0) for i in idx]

# Concatenate along dim=0
a_cat = torch.cat(a_list, dim=0)
u_cat = torch.cat(u_list, dim=0)

print(a_cat.shape, u_cat.shape)


In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from datetime import datetime, timedelta
import numpy as np

# ==========================================
# 0) Get 4 samples (unsqueeze(0) then cat)
# ==========================================
a_batch, u_batch = next(iter(eval_loader))  # u_batch: [B, nx, ny, nt] (assumed)
idx = torch.randperm(u_batch.size(0))[:4]   # choose 4 random indices from this batch

u_list = [a_batch[i].unsqueeze(0) for i in idx]   # each: [1, nx, ny, nt]
u_4 = torch.cat(u_list, dim=0)                    # [4, nx, ny, nt]

# If you also want the matching inputs:
# a_list = [a_batch[i].unsqueeze(0) for i in idx]
# a_4 = torch.cat(a_list, dim=0)

# For titles (these are indices within THIS BATCH)
batch_indices = idx.tolist()

# ==========================================
# 1) Configuration
# ==========================================
start_date = datetime(2016, 10, 5, 12, 0, 0)
time_step_seconds = 3600

depth_samples = u_4.cpu().numpy()               # [4, nx, ny, nt]
num_frames = depth_samples.shape[-1]
time_dates = [start_date + timedelta(seconds=i * time_step_seconds) for i in range(num_frames)]

vmax_global = depth_samples.max() * 0.7

# ==========================================
# 2) Initialize Figure & 2x2 Subplots
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
axes = axes.flatten()
ims = []

extent = [0, 1, 0, 1]  # keep your normalized extent

for i in range(4):
    ax = axes[i]

    z_init = depth_samples[i, :, :, 0]
    im = ax.imshow(
        z_init.T,
        extent=extent,
        origin='lower',
        cmap='terrain',
        vmin=0,
        vmax=vmax_global
    )

    ax.set_title(f"Batch sample idx {batch_indices[i]}", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.3)
    ims.append(im)

# Single colorbar
cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
fig.colorbar(ims[0], cax=cbar_ax, label='Water Depth (m)', orientation='horizontal')

plt.subplots_adjust(bottom=0.15, hspace=0.2, wspace=0.1)

# ==========================================
# 3) Update Function
# ==========================================
def update(frame_idx):
    current_dt = time_dates[frame_idx]
    fig.suptitle(
        f"Hurricane Matthew\nTime: {current_dt.strftime('%Y-%m-%d %H:%M')}",
        fontsize=16,
        fontweight='bold'
    )

    for i, im in enumerate(ims):
        z_frame = depth_samples[i, :, :, frame_idx]
        im.set_data(z_frame.T)

    return ims

# ==========================================
# 4) Animation
# ==========================================

ani = animation.FuncAnimation(fig, update, frames=range(num_frames), interval=150, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import matplotlib.pyplot as plt

# 1. Extract the specific timestep (index 30) for a specific scenario (e.g., index 0)
# DATA['discharge'] shape: [nbatch, nx, ny, nt]
scenario_idx = 0 
timestep_idx = 30
z = q_tensor[scenario_idx, ..., timestep_idx].cpu().numpy()

# 2. Define the geographic extent
# x_tensor and y_tensor are now [nx, ny] (no batch dimension)
extent = [
    x_tensor.min().item(), 
    x_tensor.max().item(), 
    y_tensor.min().item(), 
    y_tensor.max().item()
]

# 3. Create the Plot
fig, ax = plt.subplots(figsize=(10, 9))

# z.T: Transposes [nx, ny] to [ny, nx] so rows=Y and cols=X for imshow
# origin='lower': Maps the first row of data to the bottom of the plot
# cmap='Reds' for discharge intensity
im = ax.imshow(z.T, extent=extent, origin='lower', cmap='jet', vmin=0)

# 4. Formatting and Labels
fig.colorbar(im, ax=ax, label='Discharge $Q$ (m$^3$/s)', orientation='horizontal', pad=0.1)

# Display the specific simulation time from t_tensor
# t_tensor is [nt]
current_time = t_tensor[timestep_idx].item()
ax.set_title(f'Scenario {scenario_idx} - Discharge $Q$ (Step {timestep_idx})\nTime: {current_time:.1f} seconds')
ax.set_xlabel('UTM Easting (m)')
ax.set_ylabel('UTM Northing (m)')
ax.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
import os
import numpy as np
from netCDF4 import Dataset as ncDataset

def load_single_tms(file_path):
    """
    Loads an ANUGA .tms file and returns time and quantity data.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")

    # Using 'with' ensures the file is closed automatically
    with ncDataset(file_path, 'r') as nc:
        # ANUGA .tms files typically store time in a variable called 'time'
        time = nc.variables['time'][:]
        
        # List all available quantities (e.g., 'discharge', 'stage')
        available_quantities = [v for v in nc.variables.keys() if v != 'time']
        
        data = {}
        for q in available_quantities:
            data[q] = nc.variables[q][:]
            
        return time, data

# --- Example Usage ---
tms_file = "/storage/group/cxs1024/default/mehdi/dam_break/scenario_groups_1/group_000/scenario_000000/tms_files/1.tms" # Update with your actual path

try:
    time_arr, quantities = load_single_tms(tms_file)
    
    print(f"Successfully loaded: {tms_file}")
    print(f"Time steps found: {len(time_arr)}")
    print(f"Quantities available: {list(quantities.keys())}")
    
    # Example: Print the first 5 seconds and discharge values
    if 'discharge' in quantities:
        print("\nFirst 5 entries (Time | Discharge):")
        for i in range(5):
            print(f"{time_arr[i]:.2f}s | {quantities['discharge'][i]:.4f} m^3/s")

except Exception as e:
    print(f"Error loading file: {e}")

In [ ]:
plt.plot(quantities['discharge'][::900])